# 6주차 과제 베이스라인 — 이커머스 고객 세분화

실제 이커머스 데이터는 분석 목적에 따라 여러 테이블로 나뉘어 제공되는 경우가 많습니다. 이 과제 데이터도 고객정보, 거래내역, 할인정보, 세금정보, 마케팅비용의 다섯 개 테이블로 구성되어 있습니다. 이 베이스라인은 **거래내역 하나로 RFM(Recency·Frequency·Monetary)을 만들고 K=3 군집을 한 번 실행하는 가장 짧은 경로**를 제공합니다. 나머지 파일 조인과 적정 K 탐색은 선택입니다. 좋은 군집 이름보다 실행·오류·문제 분해와 다음 행동을 기록하는 일이 더 중요합니다. 거래내역 파일이 없으면 세 고객 유형을 포함한 **연습용 소규모 예시 데이터**로 전환되며, 그 군집은 실제 고객에 대한 결론이 아닙니다.


## 1. 파일 상태 확인하고 불러오기

과제 데이터는 아래 DACON 대회 페이지에서 내려받을 수 있습니다.

- 데이터 다운로드: [이커머스 고객 세분화 분석 아이디어 경진대회](https://dacon.io/competitions/official/236222)
- 권장 경로: `../dataset/extracted/이커머스 고객 세분화 분석 아이디어 경진대회/`

기본 시도에는 `Onlinesales_info.csv` 하나만 필요합니다. 나머지 네 파일이 있으면 조인까지 실행하고, 없으면 파일 상태를 알린 뒤 RFM 계산을 계속합니다. 기본 거래내역까지 없으면 연습용 데이터로 RFM·K-means 코드의 동작만 확인합니다.


In [ ]:
from pathlib import Path

import pandas as pd

base = Path('../dataset/extracted/이커머스 고객 세분화 분석 아이디어 경진대회')
paths = {
    'sales': base / 'Onlinesales_info.csv',
    'discount': base / 'Discount_info.csv',
    'customer': base / 'Customer_info.csv',
    'tax': base / 'Tax_info.csv',
    'marketing': base / 'Marketing_info.csv',
}
print('현재 작업 폴더:', Path.cwd())
for name, path in paths.items():
    print(f'{name:10s}:', '있음' if path.exists() else '없음')

if paths['sales'].exists():
    DATA_MODE = '실제 데이터'
    sales = pd.read_csv(paths['sales'])
    discount = pd.read_csv(paths['discount']) if paths['discount'].exists() else None
    customer = pd.read_csv(paths['customer']) if paths['customer'].exists() else None
    tax = pd.read_csv(paths['tax']) if paths['tax'].exists() else None
    marketing = pd.read_csv(paths['marketing']) if paths['marketing'].exists() else None
else:
    DATA_MODE = '연습용 소규모 예시 데이터'
    demo_rows = []
    profile_specs = [
        {'이름': 'RECENT', '마지막구매일': '2019-01-30', '거래수': 5, '수량': 3, '평균금액': 120.0},
        {'이름': 'MIDDLE', '마지막구매일': '2019-01-20', '거래수': 3, '수량': 2, '평균금액': 55.0},
        {'이름': 'OLD', '마지막구매일': '2019-01-05', '거래수': 1, '수량': 1, '평균금액': 18.0},
    ]
    transaction_no = 0
    for profile in profile_specs:
        for customer_no in range(6):
            customer_id = f'DEMO_{profile["이름"]}_{customer_no:02d}'
            last_date = pd.Timestamp(profile['마지막구매일']) - pd.Timedelta(days=customer_no % 2)
            for order_no in range(profile['거래수']):
                demo_rows.append({
                    '고객ID': customer_id,
                    '거래ID': f'DEMO_TX_{transaction_no:03d}',
                    '거래날짜': last_date - pd.Timedelta(days=order_no * 2),
                    '제품ID': f'DEMO_PRODUCT_{order_no % 4}',
                    '제품카테고리': ['식품', '생활', '의류', '전자'][order_no % 4],
                    '수량': profile['수량'] + customer_no % 2,
                    '평균금액': profile['평균금액'] + customer_no * 2,
                    '배송료': 5.0,
                    '쿠폰상태': 'Used' if order_no % 2 == 0 else 'Not Used',
                })
                transaction_no += 1
    sales = pd.DataFrame(demo_rows)
    discount = customer = tax = marketing = None
    print('Onlinesales_info.csv가 없어 예시 데이터로 코드 흐름만 연습합니다.')
    print('예시 군집과 Silhouette은 실제 고객 세분화 결론이 아닙니다.')

print('데이터 모드:', DATA_MODE)
print('기본 거래내역 크기:', sales.shape)


## 2. 테이블 조인

거래내역(`sales`)을 기준으로 준비된 부가 테이블만 결합합니다. 부가 파일이 없어도 다음 RFM 계산으로 이동할 수 있습니다. 각 조인 키와 결측치 처리 이유는 다음과 같습니다.

- `tax`: `제품카테고리`를 기준으로 조인합니다. 두 테이블의 카테고리가 일치하므로 조인 후 결측치가 생기지 않습니다.
- `discount`: `월`과 `제품카테고리`를 기준으로 조인합니다. 할인 테이블에 일부 카테고리가 없어 생긴 결측은 데이터 설명상 할인 없음으로 해석할 수 있을 때만 0으로 채웁니다.
- `customer`: `고객ID`를 기준으로 조인하여 고객 기본정보를 추가합니다.
- `marketing`: 거래날짜와 날짜를 기준으로 조인하여 해당 날짜의 마케팅비용을 추가합니다. 이 변수는 RFM 계산에는 사용하지 않지만, 군집을 해석할 때 참고할 수 있습니다.


In [ ]:
sales['거래날짜'] = pd.to_datetime(sales['거래날짜'])
sales['월'] = sales['거래날짜'].dt.strftime('%b')
df = sales.copy()
joined = []

if tax is not None:
    df = df.merge(tax, on='제품카테고리', how='left')
    joined.append('tax')
if discount is not None:
    df = df.merge(discount, on=['월', '제품카테고리'], how='left')
    # 데이터 설명을 확인한 이번 기본 가정: 할인 정보가 없는 카테고리는 할인율 0입니다.
    df['할인율'] = df['할인율'].fillna(0)
    joined.append('discount')
if customer is not None:
    df = df.merge(customer, on='고객ID', how='left')
    joined.append('customer')
if marketing is not None:
    marketing['날짜'] = pd.to_datetime(marketing['날짜'])
    df = df.merge(marketing, left_on='거래날짜', right_on='날짜', how='left')
    joined.append('marketing')

print('선택 조인:', joined if joined else '없음 — 거래내역만 사용합니다.')
print('조인 전/후 행 수:', len(sales), len(df))


## 3. RFM 계산

하나의 거래(`거래ID`)에 여러 상품 행(row)이 포함될 수 있습니다. 따라서 Frequency는 행의 개수가 아니라 `거래ID`의 고유값 개수로 계산합니다.


In [ ]:
df['금액'] = df['수량'] * df['평균금액']

ref_date = df['거래날짜'].max() + pd.Timedelta(days=1)

rfm = df.groupby('고객ID').agg(
    Recency=('거래날짜', lambda x: (ref_date - x.max()).days),
    Frequency=('거래ID', 'nunique'),
    Monetary=('금액', 'sum'),
).reset_index()

if customer is not None:
    rfm = rfm.merge(customer, on='고객ID', how='left')

print('관찰 기간:', df['거래날짜'].min().date(), '~', df['거래날짜'].max().date())
print('RFM 기준일:', ref_date.date())
print('고객 단위 RFM 크기:', rfm.shape)


In [ ]:
rfm.describe()


In [ ]:
rfm.head()


## 이번 주 과제 경로

- **✅ 기본 시도**: RFM에 `log1p`와 `StandardScaler`를 적용하고 K=3을 한 번 실행해 군집 크기와 평균을 확인합니다.
- **🧩 문제 분해**: 멈추면 마지막 성공 셀, 오류 마지막 줄, 의심 원인 하나, 작은 확인 코드 하나를 기록합니다. 해결하지 못해도 괜찮습니다.
- **🌱 선택 탐색**: 여유가 있을 때만 K 비교, 다른 스케일링, 부가 테이블 조인, 군집 이름 가운데 하나를 고릅니다.

기본 완료 기준은 **질문 1개 + 실행 또는 실행 시도 1개 + 관찰 결과 또는 오류 1개 + 다음 행동 1개**입니다.


### 4. ✅ 기본 시도 — 분포 확인과 스케일링

`Monetary`와 `Frequency`는 오른쪽 꼬리가 길 수 있습니다. 기본 시도에서는 값을 지우지 않고 `log1p`로 큰 값의 영향을 완화한 뒤 `StandardScaler`로 세 변수의 단위를 맞춥니다. 이 변환은 정답이 아니라 첫 선택이므로 원본 요약과 함께 기록합니다.


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

rfm_cols = ['Recency', 'Frequency', 'Monetary']
print(rfm[rfm_cols].describe().loc[['min', '50%', '75%', 'max']])

rfm_log = np.log1p(rfm[rfm_cols])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_log)
print('스케일링 결과 크기:', X_scaled.shape)
# 작은 힌트: 값의 삭제 여부를 판단하기 전에는 max가 큰 이유를 원본 거래에서 확인합니다.


### 5. ✅ 기본 시도 — K=3으로 한 번 학습

K=3은 전체 흐름을 확인하기 위한 출발점이며 적정 K라는 결론이 아닙니다. 한 번 학습하고 Silhouette Score를 확인합니다. K별 비교는 선택 탐색에서 진행합니다.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

model = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = model.fit_predict(X_scaled)
print('군집 데이터 모드:', DATA_MODE)
print('K=3 inertia:', round(model.inertia_, 2))
print('K=3 silhouette:', round(silhouette_score(X_scaled, labels), 3))


### 6. ✅ 기본 시도 — 군집 크기와 원본 RFM 평균 확인

군집 번호는 임의의 이름표입니다. 먼저 각 군집의 고객 수와 원본 단위 RFM 평균에서 차이 하나만 관찰합니다. 군집 이름은 기본 완료 조건이 아닙니다.


In [ ]:
rfm_result = rfm.copy()
rfm_result['군집'] = labels
cluster_summary = rfm_result.groupby('군집').agg(
    고객수=('고객ID', 'size'),
    Recency_평균=('Recency', 'mean'),
    Frequency_평균=('Frequency', 'mean'),
    Monetary_평균=('Monetary', 'mean'),
).round(1)
cluster_summary


## 🌱 선택 탐색 — 하나만 골라도 됩니다

아래 셀은 기본 범위에서 실행하지 않아도 됩니다. `RUN_OPTIONAL=True`로 바꾸면 K=2~6의 inertia와 Silhouette을 비교합니다. 대신 `RobustScaler`, 부가 테이블 조인 검산, 군집 이름과 검증 계획 가운데 하나를 탐색해도 됩니다. 미래 고객에게 적용하는 탐색이라면 scaler와 K-means를 기준 기간 고객에게만 fit하고 이후 고객에는 transform·predict만 수행합니다.


In [ ]:
RUN_OPTIONAL = False

if RUN_OPTIONAL:
    comparison = []
    for k in range(2, 7):
        candidate = KMeans(n_clusters=k, n_init=10, random_state=42)
        candidate_labels = candidate.fit_predict(X_scaled)
        comparison.append({
            'K': k,
            'inertia': candidate.inertia_,
            'silhouette': silhouette_score(X_scaled, candidate_labels),
        })
    print(pd.DataFrame(comparison).round(3))
else:
    print('선택 탐색을 건너뛰었습니다. 기본 시도 기록으로 이동합니다.')


## 여기까지 하면 이번 주 기록 완료

아래 네 줄을 이 셀 아래 새 Markdown 셀에 적습니다. K=3이 좋은 선택인지 결론 내리지 못했거나 실행이 멈췄어도, 오류와 다음 행동을 적었다면 완료입니다.

- 질문 1개:
- 실행 또는 실행 시도 1개:
- 관찰한 결과 또는 오류 1개:
- 다음 행동 1개:
